# The faithful-methodology battery: gemma-4-31b-it (Q1.H2.E4)

The E1/E2 nulls were on a base model, a deviation from both the paper (an assistant tuned with
reinforcement learning from human feedback, RLHF) and the reference (instruct model). This is
the corrected run: probes re-extracted from gemma-4-31b-it on the story corpus
(`results/emotion_vectors_it/`), battery activations collected under BOTH plain
`Human:/Assistant:` text and -it's own **chat template** (`results/probe_sweep_it/`),
all 20 layers x 3 readouts.

**Registered rule (unchanged from E2)**: select on the paper's 12 scenarios, count only if the
same (format, layer, readout) combination reaches target-in-top-3 >= 8/12 on the held-out 12.

**Context from the geometry check**: on -it, valence is demoted from the first principal
component (PC1, base r=0.833) to the third (PC3, r=0.762) under two larger non-affective
components. The emotion axes survive instruction tuning but no longer dominate variance.
Cosine probes use directions, not the ordering from principal component analysis (PCA), so the
battery is the behavioural test of whether those surviving directions activate on implicit
content.

In [1]:
# this cell loads -it probes and battery activations, then scores every sweep cell
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata

from emotion_vectors.probe_prompts import SCENARIOS

ROOT = Path("..")
bundle = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
emotions, layers, means = list(map(str, bundle["emotions"])), list(bundle["layers"]), bundle["means"].astype(np.float32)
sweep = np.load(ROOT / "results/probe_sweep_it/activations.npz", allow_pickle=True)
formats = list(map(str, sweep["formats"]))
prompts = [json.loads(l) for l in open(ROOT / "results/probe_sweep_it/prompts.jsonl")]
READOUTS = ["last", "mean_all", "mean_content"]
assert list(sweep["layers"]) == layers
print("formats collected:", formats)

probes_by_layer = means - means.mean(axis=0, keepdims=True)
probes_by_layer /= np.linalg.norm(probes_by_layer, axis=2, keepdims=True)
batteries = {k: [i for i, p in enumerate(prompts) if p["kind"] == k] for k in ("scenario", "heldout")}
probe_order = [t for _, t, _ in SCENARIOS]
probe_idx = [emotions.index(t) for t in probe_order]

def top3_count(fmt, kind, readout, layer_pos):
    acts = sweep[f"{fmt}_{readout}"].astype(np.float32)
    P = probes_by_layer[probe_idx, layer_pos, :]
    A = np.stack([acts[i, layer_pos] for i in batteries[kind]])
    M = P @ A.T / np.linalg.norm(A, axis=1)
    ranks = [int(12 - rankdata(M[:, j])[j]) + 1 for j in range(len(batteries[kind]))]
    return sum(r <= 3 for r in ranks), M

grids = {(fmt, kind): np.array([[top3_count(fmt, kind, r, lp)[0] for r in READOUTS]
                                for lp in range(len(layers))])
         for fmt in formats for kind in ("scenario", "heldout")}
print("grids:", {k: v.max() for k, v in grids.items()})

formats collected: ['plain', 'chat']


grids: {('plain', 'scenario'): np.int64(4), ('plain', 'heldout'): np.int64(4), ('chat', 'scenario'): np.int64(6), ('chat', 'heldout'): np.int64(4)}


## Sweep grids, both formats: no cell reaches the pass bar

**How to read**: as in notebook 04, each cell is (layer, readout); the number is scenarios with
target in top-3 of 12 (chance ~3, registered pass 8, bold). Each panel is titled with its
format (chat = the paper's construct, plain = `Human:/Assistant:` text) and its battery
(paper selection vs held-out confirmation). A combination counts only if it reaches >= 8 on
BOTH batteries of its format.

In [2]:
# this cell draws the four sweep grids with per-cell scores (bold at the pass bar of 8)
titles = [f"{fmt}: {'paper (selection)' if kind == 'scenario' else 'held-out (confirmation)'}"
          for fmt in formats for kind in ("scenario", "heldout")]
fig = make_subplots(rows=len(formats), cols=2, subplot_titles=titles,
                    shared_yaxes=True, horizontal_spacing=0.08, vertical_spacing=0.10)
for fi, fmt in enumerate(formats):
    for ki, kind in enumerate(("scenario", "heldout")):
        G = grids[(fmt, kind)]
        fig.add_trace(go.Heatmap(z=G, colorscale="Viridis", zmin=0, zmax=12,
                                 showscale=(fi == 0 and ki == 0),
                                 colorbar=dict(title="scenarios with target in top-3 (threshold 8)")),
                      row=fi + 1, col=ki + 1)
        for lp in range(len(layers)):
            for rp in range(len(READOUTS)):
                v = int(G[lp, rp])
                fig.add_annotation(x=rp, y=lp, text=f"<b>{v}</b>" if v >= 8 else str(v),
                                   showarrow=False,
                                   font=dict(size=9, color="white" if v < 7 else "black"),
                                   row=fi + 1, col=ki + 1)
        fig.update_xaxes(tickvals=list(range(len(READOUTS))), ticktext=READOUTS, tickangle=20,
                         title_text="readout", row=fi + 1, col=ki + 1)
        fig.update_yaxes(autorange="reversed", row=fi + 1, col=ki + 1)
    fig.update_yaxes(tickvals=list(range(len(layers))), ticktext=[str(l) for l in layers],
                     title_text="layer", row=fi + 1, col=1)
fig.update_layout(title="Q1.H2.E4: the faithful battery, gemma-4-31b-it probes and formats",
                  height=520 * len(formats), width=900)
fig.show()

In [3]:
# this cell ranks the top combinations and applies the registered rule
flat = [(grids[(f, "scenario")][lp, rp], grids[(f, "heldout")][lp, rp], f, layers[lp], r)
        for f in formats for lp in range(len(layers)) for rp, r in enumerate(READOUTS)]
flat.sort(reverse=True)
print("top combinations (paper, heldout, format, layer, readout):")
for row in flat[:10]:
    print(f"  paper {row[0]:2d}/12  heldout {row[1]:2d}/12  {row[2]:5s}  layer {row[3]:2d}  {row[4]}")
confirmed = [row for row in flat if row[0] >= 8 and row[1] >= 8]
print(f"\nregistered rule — combinations passing BOTH batteries at >=8/12: "
      f"{confirmed if confirmed else 'NONE'}")

top combinations (paper, heldout, format, layer, readout):
  paper  6/12  heldout  4/12  chat   layer 57  last
  paper  4/12  heldout  4/12  plain  layer 48  last
  paper  4/12  heldout  4/12  plain  layer 45  mean_all
  paper  4/12  heldout  4/12  plain  layer 27  mean_content
  paper  4/12  heldout  4/12  plain  layer 27  mean_all
  paper  4/12  heldout  4/12  chat   layer 51  mean_content
  paper  4/12  heldout  4/12  chat   layer 51  mean_all
  paper  4/12  heldout  3/12  plain  layer 54  mean_all
  paper  4/12  heldout  3/12  plain  layer 45  last
  paper  4/12  heldout  2/12  plain  layer 57  mean_content

registered rule — combinations passing BOTH batteries at >=8/12: NONE


## Reading

The verdict cell above is the registered test; TREE.md (Q1.H2.E4) records whatever it says.
E5's -it-native probes (self-generated stories and dialogues, 0-1% leakage) re-score this
battery in notebook 07.